# モジュール2 — MCPプロトコルとカスタムツールサーバー
### エージェントに「今この瞬間の情報」へのアクセスを与える

## モジュール1との違い

モジュール1のRAGは、文章の**意味を理解して検索する**のがとても得意でした。しかし
RAGには構造的にできないことが2つあります。

1. **「今この瞬間」の情報を扱えない** — 在庫数や注文状況のように、秒単位で変化するデータを
   検索対象の文書として毎回埋め込み直すのは非現実的です。
2. **「操作」ができない** — RAGは検索(読み取り)しかできません。「注文を確定する」
   「在庫を1つ減らす」のような、システムの状態を**変える**操作はRAGの仕組みの外側です。

この2つのギャップを埋めるのが **ツール** です。

> 💡 **RAGとMCPツールの違い、一言でいうと:**
> - RAG = 文章を「意味の近さ」であいまいに検索する(読み取り専用)
> - MCPツール = 正確なID・条件を指定してデータを取得・**操作**する(読み取り+書き込み)

## MCP(Model Context Protocol)とは

MCPは、AIモデルにツール・データ・プロンプトを提供するための、業界標準のオープンな
プロトコルです。標準がない世界では、AIプロバイダーごと・アプリごとに「関数呼び出し」の
独自形式を発明する必要がありました。MCPサーバーを一度作れば、MCPに対応したどんな
クライアント(モジュール3で作るエージェントも含む)からも、同じ方法でツールを見つけて
呼び出せます。

**今日作るもの、ステップごとに:**
1. 1つのツールを持つ、最小限のMCPサーバー
2. MCPクライアントでテストする(まだエージェントは登場しません。プロトコルの仕組み
   だけに集中します)
3. 2つ目のツールを追加し、入力値の検証(バリデーション)を行う
4. ツールが間違った呼び出され方をした場合の対処


## 環境セットアップ(SageMaker ノートブックインスタンス)

- **インスタンスタイプ:** `ml.t3.medium` で十分です。
- **カーネル:** `conda_pytorch_p310`
- **初回のみ:** 下のセルで、このカーネルに入っていないパッケージをインストールします。

In [ ]:
# このノートブックインスタンスで一度だけ実行してください
# conda_pytorch_p310 には fastmcp は入っていません:
%pip install --quiet 'fastmcp>=4.0'

In [ ]:
# セットアップ
import asyncio
from importlib.metadata import version, PackageNotFoundError
from fastmcp import FastMCP, Client

# MCPの仕様は2026年7月28日に大きな改訂がありました(ステートレスなプロトコルへの変更)。
# fastmcp 4.0以降が、この新しい仕様に標準対応した最初のバージョンです。この確認により、
# 古いバージョンのまま気づかずに進めてしまうことを防ぎます。
try:
    fastmcp_version = version("fastmcp")
except PackageNotFoundError:
    fastmcp_version = "0.0.0"

major_version = int(fastmcp_version.split(".")[0])
assert major_version >= 4, (
    f"fastmcp {fastmcp_version} が検出されました -- このノートブックには fastmcp>=4.0 が"
    f"必要です。実行してください: pip install --upgrade 'fastmcp>=4.0'"
)

def TODO(hint=""):
    """演習の未完成部分を示す関数です。TODO(...) の呼び出しを自分のコードに置き換えて
    ください。置き換えるまではエラーが出続けます(これは正常な動作です)。"""
    raise NotImplementedError(f"ここを実装してください。ヒント: {hint}")

print(f"準備完了です。fastmcp {fastmcp_version}")

## `async` / `await` について(初めて登場します)

このノートブックから、`async def` や `await` というキーワードが出てきます。これは
「非同期処理」と呼ばれるしくみで、MCPサーバーとの通信のように**返事が返ってくるまで
少し時間がかかる処理**を扱うためのPythonの機能です。

> 💡 **たとえ話:** カフェでコーヒーを注文したとき、コーヒーができるまでレジの前で
> 立ち止まって待つのではなく、番号札を受け取って席で待ちますよね。`await` はこの
> 「番号札を受け取って、結果が来たら受け取る」という考え方に近いものです。

今日は `async`/`await` の仕組み自体を深く理解する必要はありません。「サーバーと通信する
処理には `await` を付ける」という型を、真似しながら身につけていきましょう。

## ステップ1: ニンバス・ロボティクスの在庫データ(生きているデータ)

これが今日の「生きたデータベース」です。実際のシステムでは社内APIやデータベースへの
呼び出しになりますが、Pythonの辞書(dict)を使うことで、MCPの仕組み自体に集中できます。

モジュール1と同じ製品を使いますが、今回は**在庫数(stock)**という、時間とともに変化する
情報が追加されている点に注目してください。

In [ ]:
INVENTORY = {
    "drone-100": {"name": "ニンバス・スカウトドローン", "stock": 12, "price": 249.00},
    "drone-200": {"name": "ニンバス・カーゴドローン", "stock": 3, "price": 899.00},
}

## ステップ2: はじめてのMCPツール

MCPツールは、型ヒントとdocstring(関数の説明文)を持つ、ごく普通のPython関数です。
FastMCPはこの2つの情報から、ツールの「仕様書」を自動的に作ります。この仕様書が、
モデルに「このツールが何をするか」「どんな引数が必要か」を伝えます。

> ⚠️ **docstringは飾りではありません。** モデルは、いつこのツールを呼び出すべきかを
> **docstringを読んで判断します。** あいまいなdocstringを書くと、エージェントがツールを
> 全く呼び出さなかったり、間違ったタイミングで呼び出したりする原因になります。これは
> 実際のMCP開発で最もよくあるバグの1つです。

**あなたの番:** `check_inventory` を完成させてください。`product_id` を受け取り、
`INVENTORY` から該当する情報を返します。見つからない場合は `{"error": "..."}` のような
辞書を返してください。

In [ ]:
mcp = FastMCP("NimbusInventory")

@mcp.tool
def check_inventory(product_id: str) -> dict:
    """ニンバス・ロボティクスの製品IDを指定して、在庫数と価格を確認する。"""
    # TODO: INVENTORY から product_id を検索し、見つからなければ {"error": "..."} を返す
    TODO("dict.get(product_id) を使い、見つからない場合のフォールバックのerror辞書も用意する")


In [ ]:
# --- レスキューセル ---
mcp = FastMCP("NimbusInventory")

@mcp.tool
def check_inventory(product_id: str) -> dict:
    """ニンバス・ロボティクスの製品IDを指定して、在庫数と価格を確認する。"""
    item = INVENTORY.get(product_id)
    if not item:
        return {"error": f"Unknown product_id '{product_id}'"}
    return item

## ステップ3: MCPクライアントでサーバーと会話する

MCPサーバーをテストするために、別のプロセスを立ち上げる必要はありません。FastMCPの
`Client` は、サーバーのオブジェクトと直接メモリ上で通信できます。これはノートブックで
素早く試行錯誤するのに最適な方法です。(モジュール3や、他のどんな環境でも、同じ
クライアントのコードがstdioやHTTP経由のサーバーに対してそのまま使えます。)

**「クライアント」と「サーバー」という役割分担:**
- **サーバー**: ツールを持ち、公開する側(今作っている `mcp` オブジェクト)
- **クライアント**: サーバーに接続し、ツールを探して呼び出す側(下の `Client(mcp)`)

モジュール3で作るエージェントは、この「クライアント」の役割を担うことになります。

In [ ]:
async def list_and_call():
    async with Client(mcp) as client:
        # サーバーが公開しているツールの一覧を取得します
        tools = await client.list_tools()
        print("利用可能なツール:", [t.name for t in tools])

        # 存在する製品IDで呼び出す
        result = await client.call_tool("check_inventory", {"product_id": "drone-200"})
        print("check_inventory('drone-200') ->", result.data)

        # 存在しない製品IDで呼び出す(エラーになることを確認)
        result = await client.call_tool("check_inventory", {"product_id": "drone-999"})
        print("check_inventory('drone-999') ->", result.data)

await list_and_call()

## ステップ4: 2つ目のツールと、入力値の検証(バリデーション)

実際のツールは、不正な入力に対して防御的である必要があります。エージェント(や、
うまく設計されていないプロンプト)は、間違った引数でツールを呼び出すことがあるからです。

`place_order` というツールを追加しましょう。これは在庫を減らし、在庫が足りない場合は
注文を拒否します。

> 📌 **これはRAGにはできない「操作」です:** `check_inventory` は「読み取り」でしたが、
> `place_order` は在庫数という**システムの状態を書き換えます**。RAGの検索は何度呼んでも
> 結果は変わりませんが、この関数は呼ぶたびに在庫が減っていきます。これがMCPツールと
> RAG検索の本質的な違いです。

**あなたの番:** `place_order` を完成させてください。以下の条件を満たす必要があります:
- `product_id` が存在しない場合は `{"error": "..."}` を返す
- `quantity` が在庫数を超える場合は `{"error": "..."}` を返す
- それ以外の場合は、在庫を `quantity` だけ減らし、確認用の辞書を返す

In [ ]:
@mcp.tool
def place_order(product_id: str, quantity: int) -> dict:
    """ニンバス・ロボティクスの製品を注文する。在庫が不足している場合は失敗する。"""
    item = INVENTORY.get(product_id)
    if not item:
        return {"error": f"Unknown product_id '{product_id}'"}
    # TODO: quantity と item["stock"] を比較し、問題なければ在庫を減らして
    # 確認用の辞書(例: {"confirmed": True, "remaining_stock": ...})を返す
    TODO("quantity と item['stock'] を比較し、在庫を更新して確認用の辞書を返す")


In [ ]:
# --- レスキューセル ---
@mcp.tool
def place_order(product_id: str, quantity: int) -> dict:
    """ニンバス・ロボティクスの製品を注文する。在庫が不足している場合は失敗する。"""
    item = INVENTORY.get(product_id)
    if not item:
        return {"error": f"Unknown product_id '{product_id}'"}
    if quantity > item["stock"]:
        return {"error": f"Only {item['stock']} units of {item['name']} left in stock"}
    item["stock"] -= quantity
    return {"confirmed": True, "product": item["name"], "remaining_stock": item["stock"]}

## チェックポイント演習

次のシナリオを実行してください: カーゴドローンを2機注文(成功するはず)し、次に
さらに5機注文しようとする(在庫が残り1機しかないため失敗するはず)。これはモジュール1の
「間違った呼び方をして、何が起きるか見る」という習慣を、検索ではなくツールに対して
行うものです。

> 💭 **考えてみましょう:** 2回目の注文が失敗したとき、エラーメッセージは何を伝えて
> いますか? もしこのエラーメッセージが漠然とした「エラーが発生しました」だけだったら、
> この結果を見たAIエージェントは次にどう行動すればよいか判断できるでしょうか?

In [ ]:
async def run_scenario():
    async with Client(mcp) as client:
        tools = await client.list_tools()
        print("利用可能なツール:", [t.name for t in tools])

        r1 = await client.call_tool("place_order", {"product_id": "drone-200", "quantity": 2})
        print("注文1 (数量=2):", r1.data)

        r2 = await client.call_tool("place_order", {"product_id": "drone-200", "quantity": 5})
        print("注文2 (数量=5):", r2.data)

await run_scenario()

## まとめ

現実的な(単純化されているとはいえ)ビジネスロジックと入力値の検証を備えた、2つのツールを
持つMCPサーバーができました。モジュール3では、指示モデル(instruct model)が、
ユーザーの質問に応じて**いつこれらのツールを呼び出すか**を判断するようになります。
そして、モデルのツール呼び出しの形式がサーバーの期待する形式と少しでも食い違うと
何が起きるかも、実際に確認します。